In [1]:
import xarray as xr
import dask.array as da
import lightgbm as lgb
from dask.distributed import LocalCluster, Client, performance_report
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import psutil
import os
from dask.diagnostics import ProgressBar


cluster = LocalCluster(
    n_workers=4,  # Fewer workers = fewer WebSocket connections
    threads_per_worker=2,
    worker_dashboard_address=False  # Disable per-worker dashboards
)
client = Client(cluster)    
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 98.23 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:32887,Workers: 4
Dashboard: http://127.0.0.1:8787/status,Total threads: 8
Started: Just now,Total memory: 98.23 GiB
Comm: tcp://127.0.0.1:44971,Total threads: 2
Dashboard: http://127.0.0.1:38087/status,Memory: 24.56 GiB
Nanny: tcp://127.0.0.1:33211,


2025-04-13 22:44:35,061 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 590ce5cc33d4e27ff318ed3413b97639 initialized by task ('rechunk-merge-rechunk-transfer-720492231db60343ea37a4f2fe19b79d', 0, 0, 0, 0, 0, 0, 0, 5) executed on worker tcp://127.0.0.1:42615
2025-04-13 22:44:35,186 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 5eaf8298fb7d245b6c3db718505bfb31 initialized by task ('rechunk-merge-rechunk-transfer-720492231db60343ea37a4f2fe19b79d', 0, 1, 0, 0, 0, 1, 0, 5) executed on worker tcp://127.0.0.1:46169
2025-04-13 22:44:35,320 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 73f37b318ece866ca093185660e718ec initialized by task ('rechunk-merge-rechunk-transfer-720492231db60343ea37a4f2fe19b79d', 0, 10, 0, 0, 0, 10, 0, 5) executed on worker tcp://127.0.0.1:42615
2025-04-13 22:44:35,395 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c567f85f96f9625901c46f016b8b499c initialized by task ('rechunk-merge-rechunk-transfer-720492231db60343ea

In [2]:
from openeo.local import LocalConnection

# Initialize the local connection
local_conn = LocalConnection("./")

# Define the STAC collection URL
stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"

# Specify the spatial extent (bounding box)
spatial_extent = {
    "west": 11.0,
    "east": 12.0,
    "south": 46.0,
    "north": 47.0
}

# Specify the temporal extent
temporal_extent = ["2000-01-01", "2020-12-31"]

# Load the data cube with specified parameters
era5_single = local_conn.load_stac(
    url=stac_item,
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["data"]
).execute()

# Convert to xarray Dataset and select the 't2m' variable
era5_single = era5_single.to_dataset(dim='bands')["t2m"].to_dataset()

# Display the dataset
era5_single


<xarray.Dataset> Size: 828kB
Dimensions:  (lat: 5, lon: 5, time: 7670)
Coordinates:
  * lat      (lat) float64 40B 47.0 46.75 46.5 46.25 46.0
  * lon      (lon) float64 40B 11.0 11.25 11.5 11.75 12.0
  * time     (time) datetime64[ns] 61kB 2000-01-01 2000-01-02 ... 2020-12-30
Data variables:
    t2m      (time, lat, lon) float32 767kB dask.array<chunksize=(500, 5, 5), meta=np.ndarray>

In [3]:
stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"

from openeo.local import LocalConnection
local_conn = LocalConnection("./")

era5_pressure = local_conn.load_stac(
    url=stac_item,
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["data"]
).execute()
era5_pressure = era5_pressure.sel(lon=slice(11, 12), lat=slice(47, 46)).to_dataset(dim='bands')
era5_pressure

<xarray.Dataset> Size: 4MB
Dimensions:  (time: 7670, lat: 5, lon: 5)
Coordinates:
  * lat      (lat) float64 40B 47.0 46.75 46.5 46.25 46.0
  * lon      (lon) float64 40B 11.0 11.25 11.5 11.75 12.0
  * time     (time) datetime64[ns] 61kB 2000-01-01 2000-01-02 ... 2020-12-30
Data variables:
    q_850    (time, lat, lon) float32 767kB dask.array<chunksize=(500, 5, 5), meta=np.ndarray>
    t_850    (time, lat, lon) float32 767kB dask.array<chunksize=(500, 5, 5), meta=np.ndarray>
    u_850    (time, lat, lon) float32 767kB dask.array<chunksize=(500, 5, 5), meta=np.ndarray>
    v_850    (time, lat, lon) float32 767kB dask.array<chunksize=(500, 5, 5), meta=np.ndarray>
    z_850    (time, lat, lon) float32 767kB dask.array<chunksize=(500, 5, 5), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_edition:            1
    GRIB_subCentre:          0
    crs:                     EPSG:4326
    history:                 2024-11-15T16:05 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts

In [4]:
ERA5 = xr.merge([era5_single, era5_pressure])
ERA5

<xarray.Dataset> Size: 5MB
Dimensions:  (lat: 5, lon: 5, time: 7670)
Coordinates:
  * lat      (lat) float64 40B 47.0 46.75 46.5 46.25 46.0
  * lon      (lon) float64 40B 11.0 11.25 11.5 11.75 12.0
  * time     (time) datetime64[ns] 61kB 2000-01-01 2000-01-02 ... 2020-12-30
Data variables:
    t2m      (time, lat, lon) float32 767kB dask.array<chunksize=(500, 5, 5), meta=np.ndarray>
    q_850    (time, lat, lon) float32 767kB dask.array<chunksize=(500, 5, 5), meta=np.ndarray>
    t_850    (time, lat, lon) float32 767kB dask.array<chunksize=(500, 5, 5), meta=np.ndarray>
    u_850    (time, lat, lon) float32 767kB dask.array<chunksize=(500, 5, 5), meta=np.ndarray>
    v_850    (time, lat, lon) float32 767kB dask.array<chunksize=(500, 5, 5), meta=np.ndarray>
    z_850    (time, lat, lon) float32 767kB dask.array<chunksize=(500, 5, 5), meta=np.ndarray>

In [5]:
stac_item = "https://stac.intertwin.fedcloud.eu/collections/EMO1_TA24_PR_RG_PET_DAILY"

from openeo.local import LocalConnection
local_conn = LocalConnection("./")

emo1 = local_conn.load_stac(
    url=stac_item,
    bands=["data"],
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
).execute()
EMO1 = emo1.sel(lon=slice(11, 12), lat=slice(47, 46)).to_dataset(dim='bands')["ta24"].to_dataset()
EMO1

<xarray.Dataset> Size: 221MB
Dimensions:  (lat: 60, lon: 60, time: 7670)
Coordinates:
  * lat      (lat) float64 480B 46.99 46.98 46.96 46.94 ... 46.04 46.02 46.01
  * lon      (lon) float64 480B 11.01 11.02 11.04 11.06 ... 11.96 11.98 11.99
  * time     (time) datetime64[ns] 61kB 2000-01-01 2000-01-02 ... 2020-12-30
Data variables:
    ta24     (time, lat, lon) float64 221MB dask.array<chunksize=(365, 20, 36), meta=np.ndarray>

In [6]:
stac_item = "https://stac.intertwin.fedcloud.eu/collections/EMO1_DEM"

from openeo.local import LocalConnection
local_conn = LocalConnection("./")

dem = local_conn.load_stac(
    url=stac_item,
    spatial_extent=spatial_extent,
    bands=["data"]
).execute()
dem = dem.sel(lon=slice(11, 12), lat=slice(47, 46)).to_dataset(dim='bands')["dem"].to_dataset()
dem

<xarray.Dataset> Size: 30kB
Dimensions:  (lat: 60, lon: 60, time: 1)
Coordinates:
  * lat      (lat) float64 480B 46.99 46.98 46.96 46.94 ... 46.04 46.02 46.01
  * lon      (lon) float64 480B 11.01 11.02 11.04 11.06 ... 11.96 11.98 11.99
  * time     (time) datetime64[ns] 8B 2000-01-01
Data variables:
    dem      (time, lat, lon) float64 29kB dask.array<chunksize=(1, 60, 1), meta=np.ndarray>

In [7]:
import xarray as xr

single = xr.open_zarr("/mnt/CEPH_PROJECTS/InterTwin/Climate_Downscaling/EMO1_DOWNSCALING/data/SEAS5_AUGUST_2021_SINGLE.zarr/", chunks={}).sel(lon=slice(11, 12), lat=slice(47, 46))
pressure = xr.open_zarr("/mnt/CEPH_PROJECTS/InterTwin/Climate_Downscaling/EMO1_DOWNSCALING/data/SEAS5_AUGUST_2021_PRESSURE.zarr/", chunks={}).sel(lon=slice(11, 12), lat=slice(47, 46))
SEAS5 = xr.merge([single["t2m"], pressure])
SEAS5

<xarray.Dataset> Size: 7MB
Dimensions:  (lat: 5, lon: 5, number: 51, time: 216)
Coordinates:
  * lat      (lat) float32 20B 47.0 46.75 46.5 46.25 46.0
  * lon      (lon) float32 20B 11.0 11.25 11.5 11.75 12.0
  * number   (number) int32 204B 0 1 2 3 4 5 6 7 8 ... 43 44 45 46 47 48 49 50
  * time     (time) datetime64[ns] 2kB 2021-08-01 2021-08-02 ... 2022-03-04
Data variables:
    t2m      (time, number, lat, lon) float32 1MB dask.array<chunksize=(216, 1, 5, 5), meta=np.ndarray>
    q_850    (time, number, lat, lon) float32 1MB dask.array<chunksize=(216, 1, 5, 5), meta=np.ndarray>
    t_850    (time, number, lat, lon) float32 1MB dask.array<chunksize=(216, 1, 5, 5), meta=np.ndarray>
    u_850    (time, number, lat, lon) float32 1MB dask.array<chunksize=(216, 1, 5, 5), meta=np.ndarray>
    v_850    (time, number, lat, lon) float32 1MB dask.array<chunksize=(216, 1, 5, 5), meta=np.ndarray>
    z_850    (time, number, lat, lon) float32 1MB dask.array<chunksize=(216, 1, 5, 5), meta=np.ndarray>
Attributes:
    long_name:  2 metre temperature
    units:      K

## REMAPPING

In [8]:
import xarray as xr
import numpy as np

def match_to_mid_resolution(source_ds, target_ds, lat_name='lat', lon_name='lon', num_mid_lats=30, num_mid_lons=30):
    # Get coordinate bounds from union of source and target
    min_lat = max(source_ds[lat_name].min().item(), target_ds[lat_name].min().item())
    max_lat = min(source_ds[lat_name].max().item(), target_ds[lat_name].max().item())
    min_lon = max(source_ds[lon_name].min().item(), target_ds[lon_name].min().item())
    max_lon = min(source_ds[lon_name].max().item(), target_ds[lon_name].max().item())
    
    # Create mid-resolution grid
    mid_lats = np.linspace(min_lat, max_lat, num_mid_lats)
    mid_lons = np.linspace(min_lon, max_lon, num_mid_lons)
    
    # Interpolate both datasets to mid-resolution grid using bilinear interpolation
    mid_coords = {
        lat_name: xr.DataArray(mid_lats, dims=lat_name),
        lon_name: xr.DataArray(mid_lons, dims=lon_name)
    }
    
    source_mid = source_ds.interp(mid_coords, method='linear')
    
    return source_mid

SEAS5_mid = match_to_mid_resolution(SEAS5, dem).astype('float32')
ERA5_mid = match_to_mid_resolution(ERA5, dem).astype('float32')
EMO1_mid =  match_to_mid_resolution(EMO1, dem).astype('float32')


Normalised Min-Max

In [9]:
import xarray as xr

def normalize_dataset(ds):
    # Create a copy to avoid modifying the original dataset
    ds_normalized = ds.copy()
    
    # Loop through all data variables
    for var in ds.data_vars:
        # Subtract the minimum (along all dimensions except the variable's own)
        min_val = ds[var].min(keep_attrs=True)
        ds_normalized[var] = ds[var] - min_val
        
        # Divide by the maximum (after min subtraction)
        max_val = ds_normalized[var].max(keep_attrs=True)
        ds_normalized[var] = ds_normalized[var] / max_val
        
        # Preserve attributes if they exist
        if 'attrs' in ds[var].attrs:
            ds_normalized[var].attrs.update(ds[var].attrs)
    
    return ds_normalized

# Usage:
SEAS5_mid_normalized = normalize_dataset(SEAS5_mid)
ERA5_mid_normalized = normalize_dataset(ERA5_mid)

Cyclic Feature Addition

In [10]:
import numpy as np
import dask.array as da
from datetime import date


def encode_cyclical_features(values, max_value):
    """Encode cyclical features using sine and cosine transformations."""
    sin = np.sin(2 * np.pi * values / max_value)
    cos = np.cos(2 * np.pi * values / max_value)
    return sin, cos

def repeat_along_axis(arr, repeats, axis):
    """Repeat array along specified axis."""
    return da.repeat(arr[None, ...], repeats, axis=axis)

def get_spatial_dims(ds):
    """
    Detect spatial dimension names in the dataset.
    Returns (y_dim, x_dim) tuple based on common naming conventions.
    """
    dims = set(ds.dims)
    
    y_candidates = ['y', 'lat', 'latitude', 'lats']
    x_candidates = ['x', 'lon', 'longitude', 'long', 'lons']
    
    y_dim = next((d for d in y_candidates if d in dims), None)
    x_dim = next((d for d in x_candidates if d in dims), None)
    
    if y_dim is None or x_dim is None:
        raise ValueError(
            f"Could not detect spatial dimensions. Available dimensions: {list(dims)}. "
            f"Tried y names: {y_candidates}, x names: {x_candidates}"
        )
    
    return y_dim, x_dim

def get_existing_chunks(ds, dims):
    """
    Get chunking pattern from existing variables in the dataset.
    Returns dict of {dim: chunksize} for the specified dimensions.
    """
    chunks = {}
    for var in ds.data_vars.values():
        if hasattr(var.data, 'chunks'):
            var_chunks = dict(zip(var.dims, var.data.chunks))
            for dim in dims:
                if dim in var_chunks and dim not in chunks:
                    # Take first chunk size found for each dimension
                    chunks[dim] = var_chunks[dim][0]
        if all(dim in chunks for dim in dims):
            break
    return chunks or None

def encode_doys(ds, dim_order=('time', None, None), inplace=False):
    """
    Encode day of year as cyclical features and add to dataset,
    preserving existing chunking structure.
    
    Parameters:
    -----------
    ds : xarray.Dataset
        Input dataset containing time dimension
    dim_order : tuple, optional
        Dimension order for output arrays as (time_dim, y_dim, x_dim).
        Use None for automatic detection. Default: ('time', None, None)
    inplace : bool, optional
        If True, modify the dataset in place (default: False)
    
    Returns:
    --------
    xarray.Dataset
        Dataset with sin_doy and cos_doy variables added
    """
    
    if not inplace:
        ds = ds.copy()
    
    # Determine dimension names
    time_dim = dim_order[0] if dim_order[0] is not None else 'time'
    y_dim, x_dim = get_spatial_dims(ds) if dim_order[1] is None else (dim_order[1], dim_order[2])
    dims = (time_dim, y_dim, x_dim)
    
    # Get existing chunking pattern
    chunks = get_existing_chunks(ds, dims)
    
    # Compute day of the year
    doys = ds[time_dim].values.astype('datetime64[D]')
    doys = da.asarray([date.timetuple(doy.astype(object)).tm_yday for doy in doys])
    
    # Encode cyclical features
    sin_doy, cos_doy = encode_cyclical_features(doys, 365)
    
    # Repeat along spatial dimensions
    target_shape = tuple(len(ds[dim]) for dim in dims)
    repeat = int(np.prod(target_shape[1:]))  # y * x
    sin_doy = repeat_along_axis(sin_doy, repeat, 0).reshape(target_shape)
    cos_doy = repeat_along_axis(cos_doy, repeat, 0).reshape(target_shape)
    
    # Apply existing chunking pattern
    if chunks:
        current_chunks = {dims.index(dim): chunks[dim] for dim in dims if dim in chunks}
        sin_doy = sin_doy.rechunk(current_chunks)
        cos_doy = cos_doy.rechunk(current_chunks)
    
    # Add to dataset with attributes
    ds['sin_doy'] = (dims, sin_doy)
    ds['cos_doy'] = (dims, cos_doy)
    
    for name in ['sin_doy', 'cos_doy']:
        ds[name].attrs.update({
            'long_name': f"{'Sine' if 'sin' in name else 'Cosine'} of day of year",
            'units': 'unitless',
            'description': f"Cyclical encoding of day of year using {'sine' if 'sin' in name else 'cosine'} transform"
        })
    
    return ds

# Example usage:
encode_doys(SEAS5_mid_normalized, inplace=True)  # Modifies dataset in place
encode_doys(ERA5_mid_normalized, inplace=True)  # Modifies dataset in place


<xarray.Dataset> Size: 276MB
Dimensions:  (time: 7670, lat: 30, lon: 30)
Coordinates:
  * time     (time) datetime64[ns] 61kB 2000-01-01 2000-01-02 ... 2020-12-30
  * lat      (lat) float64 240B 46.01 46.04 46.08 46.11 ... 46.92 46.96 46.99
  * lon      (lon) float64 240B 11.01 11.04 11.08 11.11 ... 11.92 11.96 11.99
Data variables:
    t2m      (time, lat, lon) float32 28MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>
    q_850    (time, lat, lon) float32 28MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>
    t_850    (time, lat, lon) float32 28MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>
    u_850    (time, lat, lon) float32 28MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>
    v_850    (time, lat, lon) float32 28MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>
    z_850    (time, lat, lon) float32 28MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>
    sin_doy  (time, lat, lon) float64 55MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>
    cos_doy  (time, lat, lon) float64 55MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>

In [11]:
ERA5_mid_normalized

<xarray.Dataset> Size: 276MB
Dimensions:  (time: 7670, lat: 30, lon: 30)
Coordinates:
  * time     (time) datetime64[ns] 61kB 2000-01-01 2000-01-02 ... 2020-12-30
  * lat      (lat) float64 240B 46.01 46.04 46.08 46.11 ... 46.92 46.96 46.99
  * lon      (lon) float64 240B 11.01 11.04 11.08 11.11 ... 11.92 11.96 11.99
Data variables:
    t2m      (time, lat, lon) float32 28MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>
    q_850    (time, lat, lon) float32 28MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>
    t_850    (time, lat, lon) float32 28MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>
    u_850    (time, lat, lon) float32 28MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>
    v_850    (time, lat, lon) float32 28MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>
    z_850    (time, lat, lon) float32 28MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>
    sin_doy  (time, lat, lon) float64 55MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>
    cos_doy  (time, lat, lon) float64 55MB dask.array<chunksize=(500, 30, 30), meta=np.ndarray>

In [12]:
params = {"verbose": -1}


In [13]:
# Assuming your datasets are already loaded as xarray objects
# train_X: xarray Dataset with variables as features (lat, lon, time)
# train_y: xarray DataArray with target variable (lat, lon, time)
# test_X: xarray Dataset with ensemble dimension (lat, lon, time, ensemble_member)

# Align all datasets to ensure consistent coordinates
train_X, train_y = xr.align(ERA5_mid_normalized, EMO1_mid)
test_X = SEAS5_mid_normalized.reindex_like(train_X, method='nearest')

# Stack spatial dimensions for easier processing
train_X_stacked = train_X.stack(pixel=('lat', 'lon'))
train_y_stacked = train_y.stack(pixel=('lat', 'lon'))
test_X_stacked = test_X.stack(pixel=('lat', 'lon'))

In [14]:
def pixel_regression(X_pixel, y_pixel, test_data):
    """Perform LGBM regression for a single pixel"""
    # Convert to numpy arrays (this will trigger computation for this pixel)
    X = X_pixel
    y = y_pixel
    
    # Remove NaN values
    mask = ~np.isnan(y) & ~np.any(np.isnan(X), axis=1)
    X_clean = X[mask]
    y_clean = y[mask]
    
    if len(y_clean) < 10:  # Minimum samples threshold
        return np.nan * np.zeros((test_data.shape[0],))
    
    # Train-test split
    X_train, X_val, y_train, y_val = train_test_split(
        X_clean, y_clean, test_size=0.2, random_state=42
    )
    
    # LGBM model
    params = {
        'verbose': -1
    }
    
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              eval_metric='rmse')
    
    # 3. Predict for each ensemble member
    predictions = np.zeros((test_data.shape[0], test_data.shape[1]))  # [time, ensemble_members]
    
    for ens in range(test_data.shape[1]):  # Loop over ensemble members
        test_ens = test_data[:, ens, :]  # [time, features] for this ensemble
        mask_test = ~np.any(np.isnan(test_ens), axis=1)
        
        if np.sum(mask_test) > 0:
            predictions[mask_test, ens] = model.predict(test_ens[mask_test])
        else:
            predictions[:, ens] = np.nan  # If all NaN, fill with NaN

    return predictions

In [15]:
X_dask = train_X_stacked.to_array().data  # Already a Dask array
y_dask = train_y_stacked.to_array().data  # Already a Dask array
test_dask = test_X_stacked.to_array().data  # Already a Dask array

In [16]:
# Reshape for pixel processing
X_reshaped = X_dask.transpose(2, 1, 0).rechunk(chunks=(900, 7670, 8))  # (pixel, time, variable)
y_reshaped = y_dask.transpose(2, 1, 0).squeeze().rechunk(chunks=(900, 7670))     # (pixel, time)
test_reshaped = test_dask.transpose(3, 1, 2, 0).rechunk(chunks=(100, 216, 51, 8))  # (pixel, time, ensemble, variable)

In [17]:
X_reshaped = X_reshaped.persist()
y_reshaped = y_reshaped.persist()

In [18]:
# Map the regression function over all pixels
"""
results = da.map_blocks(
    lambda x, y, t: np.array([pixel_regression(x[i], y[i], t[i]) 
                              for i in range(x.shape[0])]),
    X_reshaped,
    y_reshaped,
    test_reshaped[:,:,:2,:],
    dtype=float
)
"""
results = da.map_blocks(
    lambda x, y, t: np.array([pixel_regression(x[i], y[i], t[i]) 
                    for i in range(x.shape[0])]),
    X_reshaped,  # [pixels, time, features] (ERA5)
    y_reshaped,  # [pixels, time] (EMO1)
    test_reshaped[:,:,:10,:],  # [pixels, time, 2_ensemble_members, features] (SEAS5)
    dtype=float,
    #chunks=(test_reshaped.shape[0], test_reshaped.shape[1], test_reshaped.shape[2], 1)  # [pixels, time, ensemble]
)

In [ ]:
# Compute results in parallel
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

predictions = results.compute()
predictions

/home/sdhinakaran/micromamba/envs/zarr_downScaleML/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/sdhinakaran/micromamba/envs/zarr_downScaleML/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/sdhinakaran/micromamba/envs/zarr_downScaleML/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/sdhinakaran/micromamba/envs/zarr_downScaleML/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/sdhinakaran/micromamba/envs/zarr_downScaleML/lib/python3.11/site-packages/skle